In [0]:
from pyspark.sql.functions import split, explode, col, when, sum, count, regexp_extract
from pyspark.sql.window import Window


In [0]:
data = spark.table("workspace.default.steam_game_output")

In [0]:
df = data

In [0]:
(data.count(), len(data.columns))

(55691, 465)

# Genres analysis

## What are the most represented genres ?

In [0]:
df_genre = df.withColumn(
    "genre_split",
    split(col("genre"), ",")
).withColumn(
    "genre",
    explode(col("genre_split"))
)

genre_count = (
    df_genre
    .groupBy("genre")
    .count()
    .orderBy("count", ascending=False)
)

display(genre_count)

genre,count
Indie,34271
Action,23616
Casual,12312
Adventure,11265
Adventure,10166
Strategy,10045
Casual,9774
Simulation,9629
RPG,8719
Early Access,6125


Databricks visualization. Run in Databricks to view.

In [0]:
top_genres = genre_count.limit(10)

display(top_genres)

genre,count
Indie,34271
Action,23616
Casual,12312
Adventure,11265
Adventure,10166
Strategy,10045
Casual,9774
Simulation,9629
RPG,8719
Early Access,6125


Databricks visualization. Run in Databricks to view.

#### Indie and actions were found to be most represented genres in this dataset, while RPG and simulation were less likely.

In [0]:
top_genres.write.mode("overwrite").saveAsTable("top_genres")

## Are there any genres that have a better positive/negative review ratio?


In [0]:
df_genre_sentiment = df_genre.groupBy("genre").agg(
    sum("positive").alias("positive"),
    sum("negative").alias("negative")
)

In [0]:
df_genre_sentiment = df_genre_sentiment.withColumn(
    "pos_ratio",
    col("positive") / (col("positive") + col("negative"))
)

display(
    df_genre_sentiment.orderBy("pos_ratio", ascending=False)
)

genre,positive,negative,pos_ratio
Animation & Modeling,569684,12456,0.9786030851685161
Photo Editing,577612,13674,0.9768741353592001
Design & Illustration,649643,20891,0.9688442345951137
Utilities,667050,28024,0.959681990694516
Game Development,2411,136,0.9466038476639184
Indie,5816540,509109,0.9195167167827365
Animation & Modeling,121081,13936,0.8967833680203233
Game Development,25050,3138,0.8886760323541932
Audio Production,56344,7116,0.8878663725181216
Casual,4014297,508069,0.8876541615605636


Databricks visualization. Run in Databricks to view.

#### Animation&Modelling as well as other utility softwares were found to have more positive-to-negative ratio than the rest.

In [0]:
df_genre_sentiment.write.mode("overwrite").saveAsTable("genre_sentiment")

## Do some publishers have favorite genres?


In [0]:
df_norm = df.withColumn(
    "publisher_split",
    split(col("publisher"), ",")
).withColumn(
    "genre_split",
    split(col("genre"), ",")
)

df_norm = df_norm.withColumn("publisher", explode("publisher_split")) \
                 .withColumn("genre", explode("genre_split"))

df_norm = df_norm.select("publisher", "genre")

In [0]:
counts = (
    df_norm.groupBy("publisher", "genre")
    .agg(count("*").alias("n_games"))
)

In [0]:
window = Window.partitionBy("publisher")

counts = counts.withColumn(
    "total",
    sum("n_games").over(window)
)

counts = counts.withColumn(
    "share",
    col("n_games") / col("total")
)

In [0]:
window_pub = Window.partitionBy("publisher")

heat = counts.withColumn(
    "total_games",
    sum("n_games").over(window_pub)
).withColumn(
    "genre_share",
    col("n_games") / col("total_games")
)

In [0]:
top_publishers = (
    heat.groupBy("publisher")
    .agg(sum("n_games").alias("publisher_total"))
    .orderBy(col("publisher_total").desc())
    .limit(12)
)

heat = heat.join(top_publishers.select("publisher"), on="publisher", how="inner")

display(heat)

publisher,genre,n_games,total,share,total_games,genre_share
,Indie,12,383,0.031331592689295036,383,0.031331592689295036
,Software Training,1,383,0.0026109660574412533,383,0.0026109660574412533
,Simulation,21,383,0.05483028720626632,383,0.05483028720626632
,Strategy,1,383,0.0026109660574412533,383,0.0026109660574412533
,Indie,95,383,0.24804177545691905,383,0.24804177545691905
,Violent,1,383,0.0026109660574412533,383,0.0026109660574412533
,Sexual Content,1,383,0.0026109660574412533,383,0.0026109660574412533
,Video Production,1,383,0.0026109660574412533,383,0.0026109660574412533
,Action,1,383,0.0026109660574412533,383,0.0026109660574412533
,Racing,8,383,0.020887728459530026,383,0.020887728459530026


Databricks visualization. Run in Databricks to view.

#### Big fish games has a high ratio of casual and adventure games

In [0]:
heat.write.mode("overwrite").saveAsTable("top_publishers_genre2")

## What are the most lucrative genres?

### Per genre revenue :

In [0]:
df_revenue = df.withColumn(
    "owners_low",
    regexp_extract(col("owners"), r"(\d+)", 1).cast("double")
).withColumn(
    "price_num",
    col("price").cast("double")
)

In [0]:
df_revenue = df_revenue.withColumn(
    "estimated_revenue",
    col("owners_low") * col("price_num")
)

In [0]:
df_genre_rev = df_revenue.withColumn(
    "genre",
    explode(split(col("genre"), ","))
)

In [0]:
genre_revenue = (
    df_genre_rev
    .groupBy("genre")
    .sum("estimated_revenue")
    .withColumnRenamed("sum(estimated_revenue)", "total_revenue")
    .orderBy("total_revenue", ascending=False)
)

display(genre_revenue)

genre,total_revenue
Action,8.00935267E8
Indie,6.41550439E8
Adventure,3.83822697E8
Strategy,3.59685947E8
RPG,3.57725407E8
Simulation,2.9884843E8
Adventure,2.76749874E8
Indie,1.56834464E8
Casual,1.50595226E8
Early Access,1.20796518E8


Databricks visualization. Run in Databricks to view.

In [0]:
genre_revenue.write.mode("overwrite").saveAsTable("genre_revenue")

### Per game revenue :

In [0]:
avg_revenue_per_game = (
    df_genre_rev
    .groupBy("genre")
    .avg("estimated_revenue")
    .withColumnRenamed("avg(estimated_revenue)", "avg_revenue")
    .orderBy("avg_revenue", ascending=False)
)

display(avg_revenue_per_game)

genre,avg_revenue
Web Publishing,124956.66666666667
RPG,119941.92760736197
Racing,80237.14005602241
Simulation,71097.94366197183
Strategy,70127.08941176471
Web Publishing,59683.3734939759
RPG,41028.26092441794
Adventure,37755.52793625811
Strategy,35807.46112493778
Action,33914.941861449865


Databricks visualization. Run in Databricks to view.

#### Action games were found to be the most lucrative.
#### Anyhow in a per game basis Web Publishing as well as RPG were more likely to be lucrative.

In [0]:
avg_revenue_per_game.write.mode("overwrite").saveAsTable("avg_revenue_per_game")